# Clustering Listwise DPO — Entailment-Labeled Preference Lists (100 questions)

Builds **1 good : 4 bad** preference lists where good/bad is decided **only by the binarized
entailment score** (correctness is ignored — it is only used to pick the reference trace).
Mirrors `generation/code/score_entailment.py` and `generation/code/build_entailment_list.py`.

Pipeline: prepare 100 GSM8K questions → generate 10 traces each → sort correct/wrong →
NLI entailment scoring vs the best correct trace → binarize at a threshold → build 1:4 lists.

Approximate time budget (A100; T4 is 3-5x slower on generation):

| Step | Time |
|---|---|
| Install + model download | ~5 min |
| Generate 100 q x 10 samples (256 tokens, 4-bit) | ~30-50 min |
| Process traces | seconds |
| Entailment scoring (~1000 traces, small NLI model) | ~2-5 min |
| Build 1:4 lists | seconds |

Runtime: **GPU required** (Runtime → Change runtime type → GPU).

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!nvidia-smi
!pip install -q \
    "transformers==4.45.1" \
    "accelerate==1.13.0" \
    "datasets==4.8.4" \
    bitsandbytes \
    sentencepiece \
    tqdm
print('Done.')

In [ ]:
# ── Hugging Face login (needed for Mistral-7B) ────────────────────────────────
from huggingface_hub import login
login()  # paste your HF token when prompted

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
import os, gc, json, re, random
import torch
from tqdm import tqdm

MODEL          = "mistralai/Mistral-7B-v0.1"
NLI_MODEL      = "cross-encoder/nli-deberta-v3-small"
N_QUESTIONS    = 100   # slice size
N_SAMPLES      = 10    # traces per question
MAX_NEW_TOKENS = 256
SEED           = 42
random.seed(SEED)

# entailment labeling (mirrors build_entailment_list.py)
THRESHOLD = 0.5   # score >= THRESHOLD -> good (1), else bad (0)
N_GOOD    = 1     # good traces per question (each emits its own record)
N_BAD     = 4     # ranked bad traces per record

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if DEVICE == "cuda" else "No GPU — expect slow generation")

# paths (all under /content so nothing needs a Drive mount)
QUESTIONS_PATH   = "/content/questions_100.jsonl"
RAW_TRACES_PATH  = "/content/raw_traces_100.jsonl"
PROCESSED_PATH   = "/content/processed_100.jsonl"
SCORED_PATH      = "/content/processed_100_scored.jsonl"
ENTAIL_PAIRS_PATH = "/content/entailment_pairs_100.jsonl"

print("Config ready.")

In [ ]:
# ── Utility functions (mirrors generation/code/utils.py) ─────────────────────

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

def is_correct(trace, ground_truth):
    """Ground truth appears in the last 300 chars of the trace."""
    return ground_truth.strip() in trace[-300:]

def no_similar(candidate, existing, length_window=500):
    """True when candidate differs by >=length_window chars from every existing trace."""
    for t in existing:
        if abs(len(candidate) - len(t)) < length_window:
            return False
    return True

_ERROR_PHRASES = ["error", "apolog", "i cannot", "i'm unable", "i am unable"]
def exist_error(trace):
    lowered = trace.lower()
    return any(p in lowered for p in _ERROR_PHRASES)

print("Utilities defined.")

## Step 1: Prepare questions (100-question slice)

In [ ]:
from datasets import load_dataset

def extract_gsm8k_answer(raw_answer):
    m = re.search(r"####\s*([\d,\.]+)", raw_answer)
    if m:
        return m.group(1).replace(",", "").strip()
    return raw_answer.strip()

print("Loading GSM8K train split...")
dataset = load_dataset("openai/gsm8k", "main", split="train")

questions_100 = [
    {"idx": i, "question": item["question"], "answer": extract_gsm8k_answer(item["answer"])}
    for i, item in enumerate(dataset)
][:N_QUESTIONS]

save_jsonl(questions_100, QUESTIONS_PATH)
print(f"Sliced {len(questions_100)} questions → {QUESTIONS_PATH}")

## Step 2: Generate traces (100 questions x 10 samples)

The slow step — roughly 30-50 min on A100, several hours on T4.
All `N_SAMPLES` for one question decode in a single batched `generate` call.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 4-bit quantization so the 7B model fits comfortably on T4 (16 GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL} (4-bit) for generation...")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model_gen = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb_config, device_map="auto"
)
model_gen.eval()

def build_prompt(question, tok):
    if tok.chat_template is not None:
        messages = [{"role": "user", "content": question}]
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f"Question: {question}\nAnswer:"

questions_data = load_jsonl(QUESTIONS_PATH)
raw_traces = []

for item in tqdm(questions_data, desc="generating"):
    prompt = build_prompt(item["question"], tokenizer)
    inputs = tokenizer(prompt, return_tensors="pt").to(model_gen.device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model_gen.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=1.0,
            num_return_sequences=N_SAMPLES,
            pad_token_id=tokenizer.eos_token_id,
        )

    for seq in output_ids:
        trace = tokenizer.decode(seq[prompt_len:], skip_special_tokens=True)
        raw_traces.append({
            "idx": item["idx"],
            "question": item["question"],
            "answer": item["answer"],
            "trace": trace,
        })

save_jsonl(raw_traces, RAW_TRACES_PATH)
print(f"Saved {len(raw_traces)} raw traces → {RAW_TRACES_PATH}")

# free VRAM before NLI scoring
del model_gen
gc.collect()
torch.cuda.empty_cache()

## Step 3: Process traces (sort correct / wrong, dedup disabled)

In [ ]:
from collections import defaultdict

raw = load_jsonl(RAW_TRACES_PATH)
pools = defaultdict(lambda: {"question": "", "answer": "", "correct_solutions": [], "wrong_solutions": []})

for item in tqdm(raw, desc="processing"):
    idx, trace, gt = item["idx"], item["trace"], item["answer"]
    pool = pools[idx]
    pool["question"] = item["question"]
    pool["answer"]   = gt

    if is_correct(trace, gt):
        if not exist_error(trace) and no_similar(trace, pool["correct_solutions"], length_window=0):
            pool["correct_solutions"].append(trace)
    else:
        if no_similar(trace, pool["wrong_solutions"], length_window=0):
            pool["wrong_solutions"].append(trace)

results = [{"idx": idx, **pool} for idx, pool in pools.items()]
save_jsonl(results, PROCESSED_PATH)

n_no_correct = sum(1 for r in results if not r["correct_solutions"])
print(f"Saved {len(results)} questions → {PROCESSED_PATH}")
print(f"Questions with zero correct traces (will be dropped later): {n_no_correct}")

## Step 4: Entailment scoring (mirrors `score_entailment.py`)

Every trace (correct and wrong) is scored with an NLI cross-encoder against the
**reference** = shortest correct trace of its question. Score = P(entailment).

In [ ]:
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification

# Traces end with the answer, so the tail carries the most signal;
# also keeps inputs within the 512-token limit.
TAIL_CHARS = 800
NLI_BATCH  = 32

print(f"Loading NLI model {NLI_MODEL}...")
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE)
nli_model.eval()

entail_idx = next(
    (i for i, l in nli_model.config.id2label.items() if l.lower() == "entailment"), 2
)

def score_pairs(premises, hypotheses):
    """Return P(entailment) for each (premise, hypothesis) pair."""
    scores = []
    for i in range(0, len(premises), NLI_BATCH):
        enc = nli_tokenizer(
            premises[i : i + NLI_BATCH],
            hypotheses[i : i + NLI_BATCH],
            truncation=True, max_length=512, padding=True, return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            logits = nli_model(**enc).logits
        probs = F.softmax(logits, dim=-1)
        scores.extend(probs[:, entail_idx].cpu().tolist())
    return scores

scored = []
for item in tqdm(load_jsonl(PROCESSED_PATH), desc="scoring"):
    corrects, wrongs = item["correct_solutions"], item["wrong_solutions"]

    if not corrects:  # no reference -> nothing meaningful to score
        scored.append({**item, "entailment_scores": {"correct": [], "wrong": [0.0] * len(wrongs)}})
        continue

    reference = min(corrects, key=len)
    ref_tail  = reference[-TAIL_CHARS:]

    if len(corrects) == 1:
        correct_scores = [1.0]
    else:
        correct_scores = score_pairs([ref_tail] * len(corrects), [c[-TAIL_CHARS:] for c in corrects])

    wrong_scores = score_pairs([ref_tail] * len(wrongs), [w[-TAIL_CHARS:] for w in wrongs]) if wrongs else []

    scored.append({**item, "entailment_scores": {"correct": correct_scores, "wrong": wrong_scores}})

save_jsonl(scored, SCORED_PATH)
print(f"Saved scored results → {SCORED_PATH}")

# score distribution — use this to sanity-check THRESHOLD before Step 5
all_scores = [s for r in scored if r["correct_solutions"]
              for s in r["entailment_scores"]["correct"] + r["entailment_scores"]["wrong"]]
print(f"\n{len(all_scores)} scores | min {min(all_scores):.3f} | max {max(all_scores):.3f}")
for lo in [i / 10 for i in range(10)]:
    n = sum(1 for s in all_scores if lo <= s < lo + 0.1)
    print(f"  [{lo:.1f}-{lo + 0.1:.1f}) {'#' * (60 * n // max(1, len(all_scores)))} {n}")

## Step 5: Build 1:4 entailment-labeled lists (mirrors `build_entailment_list.py`)

Correct and wrong pools are **merged**; the only label is `score >= THRESHOLD`.
A wrong-answer trace with high entailment counts as good, and vice versa.
Output is `ListwiseTrainer`-compatible (`prompt`, `chosen`, `rejected1-4`;
the `*_score` fields are debugging metadata the trainer ignores).

If the skip counts below are high, revisit `THRESHOLD` using the histogram above and re-run this cell.

In [ ]:
pairs = []
skipped_no_reference = 0
skipped_ratio = 0

for item in tqdm(load_jsonl(SCORED_PATH), desc="building entailment lists"):
    if not item["correct_solutions"]:
        skipped_no_reference += 1
        continue

    es = item["entailment_scores"]
    pooled = list(zip(item["correct_solutions"], es["correct"])) + \
             list(zip(item["wrong_solutions"],   es["wrong"]))

    goods = sorted((p for p in pooled if p[1] >= THRESHOLD), key=lambda x: x[1], reverse=True)
    bads  = sorted((p for p in pooled if p[1] < THRESHOLD),  key=lambda x: x[1], reverse=True)

    if len(goods) < N_GOOD or len(bads) < N_BAD:
        skipped_ratio += 1
        continue

    ranked_bads = bads[:N_BAD]
    for chosen, chosen_score in goods[:N_GOOD]:
        record = {"prompt": item["question"], "chosen": chosen}
        for i, (bad, _) in enumerate(ranked_bads, start=1):
            record[f"rejected{i}"] = bad
        record["chosen_score"]    = chosen_score
        record["rejected_scores"] = [s for _, s in ranked_bads]
        pairs.append(record)

save_jsonl(pairs, ENTAIL_PAIRS_PATH)
print(f"Skipped {skipped_no_reference} questions (no correct solution → no reference)")
print(f"Skipped {skipped_ratio} questions (fewer than {N_GOOD} good or {N_BAD} bad at threshold {THRESHOLD})")
print(f"Saved {len(pairs)} preference lists ({N_GOOD}:{N_BAD} good:bad) → {ENTAIL_PAIRS_PATH}")

## Step 6: Inspect a sample and download the outputs

In [ ]:
for r in load_jsonl(ENTAIL_PAIRS_PATH)[:1]:
    print("prompt:      ", r["prompt"][:120], "...")
    print("chosen:      ", r["chosen"][:120].replace("\n", " "), "...")
    print("chosen_score:", round(r["chosen_score"], 3))
    print("rejected_scores:", [round(s, 3) for s in r["rejected_scores"]])

try:
    from google.colab import files
    files.download(ENTAIL_PAIRS_PATH)
    files.download(SCORED_PATH)
except ImportError:
    print("Not running on Colab — files are at:", ENTAIL_PAIRS_PATH, SCORED_PATH)

## Next steps

- Train with the repo's listwise trainer:
  `python training/train_listwise.py --config training/configs/listwise_config.yaml --dataset_path entailment_pairs_100.jsonl --output_dir outputs/listwise_entailment_100`
  (or reuse the training cells from `smoke_test_colab.ipynb` — the output schema is identical).
- Compare against the correctness-labeled baseline built by `generation/code/build_listwise.py` on the same traces.